# 07 - Word Documents (DOCX) Ingestion & Parsing

> Complete guide and in-depth Arabic documentation: [`README.md`](./README.md)

Extracts structured content, headings, and tables from `.docx` files.

## 1. Ingestion with Docx2txtLoader (Plain Text)

In [ ]:
import os
from pathlib import Path

# استيراد اللوادر الخاصة بمستندات Word من LangChain
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("LangChain & Document Loaders imported successfully!")

## 2. Ingestion with UnstructuredWordDocumentLoader
Parses full document structure including tables and lists.

In [ ]:
from pathlib import Path
data_dir = Path('data') if Path('data').exists() else Path('../../data')
sample_path = (data_dir / 'docx' / 'proposal.docx').resolve()

if sample_path.exists():
    print(f"[+] File found: {sample_path} (Size: {sample_path.stat().st_size} bytes)")
else:
    print(f"[-] File not found at {sample_path}!")


## 3. Element-Level Parsing (mode='elements')
Extracts categorized structural elements (Title, NarrativeText, Table, ListItem).

In [ ]:
try:
    docx_loader = Docx2txtLoader(str(sample_path))
    docs = docx_loader.load()

    print(f"[+] Total Documents Loaded: {len(docs)}")
    print(f"[+] Metadata: {docs[0].metadata}")
    print(f"[+] Total Characters: {len(docs[0].page_content)}")
    print("\n--- First 300 Characters Preview ---")
    print(docs[0].page_content[:300])
except Exception as e:
    print(f"[-] Error loading document: {e}")

## 4. Table Extraction as Markdown Tables

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " "]
)

chunks = splitter.split_documents(docs)
print(f"[+] Document successfully split into {len(chunks)} chunks!\n")

for i, chunk in enumerate(chunks, 1):
    print(f"--- Chunk #{i} (Length: {len(chunk.page_content)}) ---")
    print(chunk.page_content.strip())
    print()

## 5. Document Splitting with Element Preservation

In [ ]:
try:
    from langchain_community.document_loaders import UnstructuredWordDocumentLoader
    
    unstructured_loader = UnstructuredWordDocumentLoader(str(sample_path), mode="elements")
    unstructured_docs = unstructured_loader.load()

    print(f"[+] Total Structured Elements Loaded: {len(unstructured_docs)}\n")

    for idx, element in enumerate(unstructured_docs[:6], 1):
        category = element.metadata.get("category", "Unknown")
        print(f"--- Element #{idx} | Category: [{category}] ---")
        print(f"Content: {element.page_content}")
        print(f"Metadata: {element.metadata}\n")
except Exception as e:
    print(f"[-] Notice: UnstructuredWordDocumentLoader encountered an environment exception: {e}")
    print("[-] Tip: Ensure 'unstructured' and its native NLP dependencies are installed.")

## 6. Strategy Comparison: Docx2txt vs Unstructured

In [ ]:
import docx

def extract_docx_tables_to_markdown(docx_file: str):
    doc = docx.Document(docx_file)
    markdown_tables = []
    
    for table_idx, table in enumerate(doc.tables, 1):
        rows_data = []
        for row in table.rows:
            row_text = [cell.text.strip().replace("\n", " ") for cell in row.cells]
            rows_data.append(row_text)
            
        if not rows_data:
            continue
            
        # بناء جدول Markdown
        header = "| " + " | ".join(rows_data[0]) + " |"
        separator = "| " + " | ".join(["---"] * len(rows_data[0])) + " |"
        body_rows = ["| " + " | ".join(row) + " |" for row in rows_data[1:]]
        
        md_table = "\n".join([header, separator] + body_rows)
        markdown_tables.append(md_table)
        
    return markdown_tables

tables = extract_docx_tables_to_markdown(str(sample_path))
print(f"[+] Extracted {len(tables)} tables as Markdown:\n")
for t_idx, md_table in enumerate(tables, 1):
    print(f"--- Table #{t_idx} ---")
    print(md_table)
    print()

## Summary
Use Docx2txt for fast plain text, and Unstructured when document hierarchy and tables are critical.